In [ ]:
#installing kaggle library
!pip install kaggle

In [ ]:
#upload your kaggle api key
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_ac9df80f4fc2204a19d5ad99bca21fd4'



Importing Twitter Sentiment Dataset

In [ ]:
!kaggle datasets download -d kazanova/sentiment140

In [ ]:
# Extracting the compressed datset

from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset,'r') as zip: #r is for read , zip is temporary name of zip file that we want to extract.
  zip.extractall()
  print('The dataset is extracted')

Importing the Dependencies

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
#printing the stopwords of english
print(stopwords.words('english'))

In [ ]:
# loading the data from csv file to pandas dataframe
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1')

In [ ]:
# checking the number of rows and columns
twitter_data.shape

In [ ]:
# printing the first 5 rows of dataframe
twitter_data.head()

In [ ]:
# naming the columns and reading the dataset again

column_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', names = column_names)

In [ ]:
twitter_data.shape

In [ ]:
twitter_data.head()

In [ ]:
#counting the number of missing values in the dataset in each column
twitter_data.isnull().sum()

In [ ]:
# checking the distribution of target column
twitter_data['target'].value_counts()

Convert the positive target value "4" to "1"

In [ ]:
twitter_data.replace({'target':{4:1}}, inplace=True) # inplace = true for changing in original dataset

In [ ]:
# checking the distribution of target column
twitter_data['target'].value_counts()

0 ---> Negative Tweet

1 ---> Positive Tweet

**Stemming**

Stemming is the process of converting word into base form by removing suffixes values

example : actor , acrtess , acting = act

In [ ]:
port_stem = PorterStemmer()

In [ ]:
stop_words = set(stopwords.words('english'))

def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content) # content = tweet , ^ = remove word if word not contain a-zA-Z alphabets ****
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split() # split words based on whitespace
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stop_words]
  stemmed_content = ' '.join(stemmed_content) # join words together
  return stemmed_content

Creating new column stemmed content

In [ ]:
twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming) # 50 minutes to complete this operation

In [ ]:
twitter_data.head()

In [ ]:
print(twitter_data['stemmed_content'])

In [ ]:
print(twitter_data['target'])

In [ ]:
# sepearating the data and label
X = twitter_data['stemmed_content'].values # data
Y = twitter_data['target'].values # label

In [ ]:
print(X)

In [ ]:
print(Y)

Splitting the data to training data and test data

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify = Y, random_state = 2)
# 0.2 means test data is 20% and rest 80% is training data.
# stratify means negative and positive equally distribute. trainging and testing both contain 80:20 % ratio.

In [ ]:
print(X.shape , X_train.shape, X_test.shape)

In [ ]:
print(X_train)

In [ ]:
print(X_test)

In [ ]:
# converting the textual data to numerical data

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
print(X_train)

In [ ]:
print(X_test)

Training the Machine Learning Model

Logistic Regression

In [ ]:
model = LogisticRegression(max_iter = 1000)

In [ ]:
model.fit(X_train, Y_train)

Model Evaluation

Accuracy Score

In [ ]:
# Accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score( Y_train, X_train_prediction)

In [ ]:
print("Accuracy Score on the training data: ", training_data_accuracy)

In [ ]:
# Accuracy score on the test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score( Y_test, X_test_prediction)

In [ ]:
print("Accuracy Score on the training data: ", test_data_accuracy)

Model Accuracy = 77.6 %

Saving the trained model

In [ ]:
import pickle

In [ ]:
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb')) # wb means writing and creating new file b means writing in binary file

Using the saved model for future predictions

In [ ]:
# loading the saved model
loaded_model = pickle.load(open('/content/trained_model.sav', 'rb')) # rb means reading and creating new file b means reading in binary file

In [ ]:
X_new = X_test[200]
print(Y_test[200])

prediction = loaded_model.predict(X_new)
print(prediction)

if (prediction[0] == 0):
  print('The tweet is negative')
else:
  print('The tweet is positive')


In [ ]:
X_new = X_test[3]
print(Y_test[3])

prediction = loaded_model.predict(X_new)
print(prediction)

if (prediction[0] == 0):
  print('The tweet is negative')
else:
  print('The tweet is positive')